In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx

import numpy as np

from datetime import datetime
from time import time
import copy

In [2]:
from onnx2pytorch import ConvertModel

In [3]:
import json_to_pytorch

In [4]:
def get_loaders(batch_size=64, num_workers=2, normalize=False):
    # Statistiche CIFAR-10 standard
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    if normalize:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
    else:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
      ])

    train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [5]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

In [6]:
test_model = json_to_pytorch.json_to_pytorch("../results/gelu/best_model_bn_8_0.0001l1_no_pad_50.json", double_precision=True)

In [7]:
test_model

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
  (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
  (6): ChebyshevPoly()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Linear(in_features=288, out_features=256, bias=True)
  (9): ChebyshevPoly()
  (10): Linear(in_features=256, out_features=10, bias=True)
)

In [8]:
# Import onnx model
onnx_model = onnx.load("../networks/cifar/best_model_bn_8_0.0001l1_no_pad.onnx")
onnx.checker.check_model(onnx_model)
# convert it to pytorch
reference_model = ConvertModel(onnx_model)

In [9]:
train_loader, test_loader = get_loaders(batch_size=128)

/home/samuel/Dokumente/Projects/NNV/FHE/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [10]:
next(iter(test_loader))[0].shape

/home/samuel/Dokumente/Projects/NNV/FHE/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([128, 3, 32, 32])

In [11]:
reference_loss,  reference_acc  = evaluate(reference_model, test_loader, criterion, "cpu")
print(f"Reference model - Loss: {reference_loss:.4f}, Accuracy: {reference_acc:.4f}")

Reference model - Loss: 0.6718, Accuracy: 0.7728


In [12]:
test_loss,  test_acc  = evaluate(test_model, test_loader, criterion, "cpu")
print(f"Test model - Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")

Test model - Loss: 2.8736, Accuracy: 0.3006


In [13]:
# Get sample from test_loader
test_input, test_label = next(iter(test_loader))

In [14]:
#test_input = test_input[0:1,:,:,:]

In [15]:
import attack

In [16]:
reference_outputs = reference_model(test_input)
test_outputs = test_model(test_input)
# Count differences in predictions
print(f"Number of different predictions: {(reference_outputs.argmax(1) != test_outputs.argmax(1)).sum().item()}")

Number of different predictions: 94


In [17]:
adv = attack.Adversary(attacks_iter=10, attack_restarts=10, attack_epsilon=2.0/255.0)

In [18]:
robust_eval_results = attack.robust_eval(reference_model, test_model, test_loader, torch.nn.CrossEntropyLoss(), "cpu", adv)

In [19]:
robust_eval_results

{'normal_loss_ref': 0.0052948335647583005,
 'normal_loss_test': 0.022582920778135035,
 'adv_loss_ref': 0.006172868412733078,
 'adv_loss_test': 0.023042070718082664,
 'normal_acc_ref': 0.7728,
 'normal_acc_test': 0.3006,
 'adv_acc_ref': 0.6684,
 'adv_acc_test': 0.1931,
 'normal_eq': 0.305,
 'adv_eq': 0.0365,
 'total': 10000}